# Analyse Sentinel-1 backscatter changes over Kverkjökull using Google Earth Engine

In [ ]:
# System
import os
import sys

# Basic
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Geospatial data
import geopandas as gpd
from shapely.geometry import Point, Polygon

# Google Earth Engine
import ee
import geemap

In [ ]:
# Uncomment if you need to re-authenticate an expired token.
ee.Authenticate()

# Initialize the Earth Engine.
ee.Initialize(project="YOUR-PROJECT-NAME")

## Load the RGI shapefiles of Iceland's glaciers and find Kverkjökull

In [ ]:
rgi = gpd.read_file("data/RGI/RGI2000-v7.0-G-06_iceland.shp")
glacier = rgi.loc[rgi.rgi_id == "RGI2000-v7.0-G-06-00373"]

## Create Sentinel-1 mosaics using Google Earth Engine

In [ ]:
def s1_mosaics(gdf, year, polarization="VH"):
    """
    Create orbit mosaics of Sentinel-1 scenes between 1st of January and 1st of April for the desired year.
    The formulation follows Guillet et al. (2025) https://doi.org/10.1017/jog.2025.10065
    """
    
    # The area of interest (bounding box with 5% padding).
    bbox = gdf.to_crs(gdf.estimate_utm_crs()).buffer(200).to_crs(gdf.crs).envelope
    west, south, east, north = bbox.total_bounds
    aoi = ee.Geometry.BBox(west=west, south=south, east=east, north=north)
    
    # Find the satellite images for the bounding box.
    scenes = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(f"{year}-01-01", f"{year}-04-01")
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', polarization))
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
    )

    mosaics = pd.DataFrame(columns=["relativeOrbitNumber", "mosaic"])

    for i, ron in enumerate(scenes.distinct("relativeOrbitNumber_start").aggregate_array("relativeOrbitNumber_start").getInfo()):
        mosaics.loc[i, "relativeOrbitNumber"] = ron
        mosaics.loc[i, "mosaic"] = scenes.filter(
            ee.Filter.eq("relativeOrbitNumber_start", ron)
        ).select(polarization).max().clip(aoi)
    
    return mosaics

In [ ]:
def orb_diff_years(gdf, y1, y2):
    """
    Compute relative backscatter difference for each Relative Orbit Number (RON).
    """
    mosaics1 = s1_mosaics(gdf=gdf, year=y1)
    mosaics2 = s1_mosaics(gdf=gdf, year=y2)

    rons = [ron for ron in mosaics1.relativeOrbitNumber.values if ron in mosaics2.relativeOrbitNumber.values]

    diffs = pd.DataFrame(columns=["year1", "year2", "relativeOrbitNumber", "diff"])
    
    def ndi(m1, m2):
        return m1.subtract(m2).divide(m1.add(m2))

    for i, ron in enumerate(rons):
        diffs.loc[i, ["year1", "year2", "relativeOrbitNumber", "diff"]] = [
            y1, y2, ron, ndi(
                mosaics1.loc[mosaics1.relativeOrbitNumber == ron, "mosaic"].values[0], 
                mosaics2.loc[mosaics2.relativeOrbitNumber == ron, "mosaic"].values[0]
            )
        ]

    return diffs

In [ ]:
def join_df(df1, df2):
    """
    Join two DataFrames (works also for GeoDataFrames). The two DataFrames must have the same columns.

    :param df1: The first DataFrame.
    :param df2: The second DataFrame.
    :return: The combined DataFrame.
    """

    def check_dfs():
        if len(df1.columns) != len(df2.columns):
            print("df1", df1.columns)
            print("df2", df2.columns)
            raise ValueError("The columns of the two DataFrames do not match.")
        elif set(df1.columns) != set(df2.columns):
            print("df1", df1.columns)
            print("df2", df2.columns)
            raise ValueError("The columns of the two DataFrames do not match.")

    if len(df1) == 0:
        return df2
    elif len(df2) == 0:
        return df1
    else:
        check_dfs()
        return pd.concat([df1, df2], ignore_index=True)

In [ ]:
def orb_diff_years(gdf, years=[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]):
    """
    Collect the relative backscatter differences for all orbits for all the specified years.
    """
    df = pd.DataFrame()

    for i, year in enumerate(years):
        df = join_df(df1=df, df2=orb_diff_years(gdf=gdf, y1=year, y2=year+1))

    return df

## Compute all the backscatter differences and visualise the results

In [ ]:
dfs = orb_diff_times(gdf=glacier)

In [ ]:
orb = 9  # 111, 147, 9, (118), 45
smooth = True

dfs_sel = dfs.loc[dfs.relativeOrbitNumber==orb]

Map = geemap.Map(center=(64.684721, -16.628830), zoom=11)
# Map.centerObject(dfs_sel.iloc[0]["diff"], zoom=10)

for i in dfs_sel.index:
    if smooth:
        Map.addLayer(
            dfs_sel.loc[i, "diff"].focal_median(100, "circle", "meters"), 
            {"min": -0.3, "max": 0.3}, 
            f"NDI Orb:{orb} {dfs_sel.loc[i, 'year1']}-{dfs_sel.loc[i, 'year2']}"
        )
    else:
        Map.addLayer(dfs_sel.loc[i, "diff"], {"min": -0.3, "max": 0.3}, f"NDI_diff Orb:{orb} {dfs_sel.loc[i, 'year1']}-{dfs_sel.loc[i, 'year2']}")

Map

## Export results to Google Drive

In [ ]:
def export_image_toDrive(gdf, image, name, folder="kverkjokull", scale=40, crs="EPSG:4326", fileFormat="GeoTIFF"):
    """
    Export a specific GEE image to a folder in the user's Google Drive account.
    """
    # The area of interest (bounding box with 5% padding).
    bbox = gdf.to_crs(gdf.estimate_utm_crs()).buffer(200).to_crs(gdf.crs).envelope
    west, south, east, north = bbox.total_bounds
    aoi = ee.Geometry.BBox(west=west, south=south, east=east, north=north)
    
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=name,
        scale=scale,
        region=aoi,
        folder=folder,
        fileNamePrefix=name,
        crs=crs,
        fileFormat=fileFormat
    )
    # Execute the task.
    task.start()

    return task

In [ ]:
# For example, saving a smoothed image (using 100m kernel) for a specified year from the selected orbit.
year=2023
export_image_toDrive(
    gdf=glacier,
    image=dfs_sel.loc[dfs_sel.year1 == year, "diff"].values[0].focal_median(100, "circle", "meters"),
    name=f"orb{orb}_{year}-{year+1}_smooth"
)